# Modèle 4 — Chatbot RAG

Construit et évalue le pipeline de récupération-génération augmentée (RAG) :

| étape | outil | sortie |
|-------|-------|--------|
| **Base de connaissances** | résumés texte par ligne, compagnie, anomalie | ~1 300 documents |
| **Index vectoriel** | sentence-transformers + ChromaDB | `data/processed/chroma_kb/` |
| **Réponses** | Llama 3 via Groq API | réponses en langage naturel |

Modifiez le bloc **Config** pour changer le modèle LLM ou le nombre de documents récupérés.

In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parents[1]))

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parents[1] / '.env')

from src.data import rag
from src.data import anomaly as an

print('imports OK')

In [ ]:
# ── Config — modifiez ici ──────────────────────────────────────────────────
EMBED_MODEL = 'all-MiniLM-L6-v2'   # modèle d'encodage sentence-transformers
TOP_K       = 5                     # documents récupérés par requête
LLM_MODEL   = 'llama3-8b-8192'     # modèle Groq (llama3-8b-8192 ou llama3-70b-8192)

# Chemins
ROOT       = Path.cwd().parents[1]
FOUNDATION = ROOT / 'data' / 'processed' / 'foundation_arrivals_full.parquet'
LINE_DIST  = ROOT / 'data' / 'processed' / 'line_distances.parquet'
CHROMA_DIR = ROOT / 'data' / 'processed' / 'chroma_kb'

print(f'foundation : {FOUNDATION.exists()}')
print(f'line_dist  : {LINE_DIST.exists()}')
print(f'index chroma dans : {CHROMA_DIR}')

In [ ]:
# Charger les données source
fa        = pd.read_parquet(FOUNDATION)
fa['trip_start'] = pd.to_datetime(fa['trip_start'])
fa['arrival']    = pd.to_datetime(fa['arrival'])
line_dist = pd.read_parquet(LINE_DIST)

print(f'foundation : {len(fa):,} lignes | lignes : {line_dist.shape[0]}')

In [ ]:
# Calculer les anomalies (nécessaires pour les documents d'anomalie)
CFG   = an.AnomalyConfig()
trips = an.trip_features(fa, CFG)
if_model, if_mean, if_std = an.train_isolation_forest(trips, CFG)
trips_scored = an.score_trips(if_model, if_mean, if_std, trips)
anomalies    = trips_scored[trips_scored['anomaly']].reset_index(drop=True)

print(f'anomalies détectées : {len(anomalies)}')
display(anomalies[['day','line','societe','dir','max_dwell_s','if_score']].head(5))

In [ ]:
# Générer les documents texte pour la base de connaissances
docs, ids, metas = rag.build_knowledge_base(fa, line_dist, anomalies)

print(f'documents générés : {len(docs)}')
print(f'  docs lignes      : {sum(1 for m in metas if m["type"]=="line")}')
print(f'  docs compagnies  : {sum(1 for m in metas if m["type"]=="company")}')
print(f'  docs anomalies   : {sum(1 for m in metas if m["type"]=="anomaly")}')
print()
print('Exemples de documents :')
for d in docs[:3]: print(' -', d)

In [ ]:
# Construire l'index vectoriel ChromaDB
col, embed_model = rag.build_chroma(docs, ids, metas, CHROMA_DIR)

print(f'index ChromaDB prêt : {col.count()} vecteurs')
print(f'modèle d\'encodage : {EMBED_MODEL}')
print(f'persisté dans : {CHROMA_DIR}')

In [ ]:
# Tester la récupération (avant le LLM)
test_queries = [
    'Quelle compagnie a le pire taux de correspondance GPS ?',
    'Quelle est la longueur de la ligne 209 ?',
    'Y a-t-il eu des anomalies sur la ligne 217 ?',
]
for q in test_queries:
    print(f'Q: {q}')
    hits = rag.retrieve(q, col, embed_model, k=TOP_K)
    for i, h in enumerate(hits, 1):
        print(f'  [{i}] {h}')
    print()

In [ ]:
# Pipeline RAG complet — poser des questions opérateur
questions = [
    'Quelle compagnie exploite le plus de lignes ?',
    'Quel est le taux de correspondance GPS de chaque compagnie ?',
    'Y a-t-il eu des trajets anormaux sur la ligne 217 ? Que s\'est-il passé ?',
    'Quelle est la ligne la plus longue du réseau ?',
    'Quelles lignes n\'ont pas de géométrie d\'arrêts ?',
]

rag_results = []
for q in questions:
    result = rag.ask(q, col, embed_model, k=TOP_K)
    display(Markdown(f'**Q :** {q}'))
    display(Markdown(f'**R :** {result["answer"]}'))
    print(f'  [jetons : {result["tokens_used"]} | docs contexte : {len(result["context"])}]\n')
    rag_results.append({'Question': q, 'Tokens': result['tokens_used']})

In [ ]:
# ── Grid Search — TOP_K de récupération ───────────────────────────────────────
# Sweeps TOP_K (number of retrieved documents per query) without making LLM API calls.
# Metrics:
#   coverage  — avg docs retrieved per query (higher TOP_K → trivially higher)
#   uniqueness — fraction of docs that are distinct (lower at high K = redundant retrieval)
#   score     — combined: balance of coverage vs uniqueness
# Updates TOP_K so the downstream evaluation cell uses the optimal value.

eval_questions_gs = [
    'Quelle compagnie a le pire taux de correspondance GPS ?',
    'Quelle est la longueur de la ligne 209 ?',
    'Y a-t-il eu des anomalies sur la ligne 217 ?',
    'Quelle compagnie exploite le plus de lignes ?',
    'Quelles lignes n\'ont pas de géométrie d\'arrêts ?',
]

topk_grid    = [2, 3, 5, 7, 10, 15]
topk_results = []

print(f'Evaluating TOP_K ∈ {topk_grid} (retrieval only, no LLM calls) ...\n')

for k in topk_grid:
    all_hits, unique_snippets = [], set()
    for q in eval_questions_gs:
        hits = rag.retrieve(q, col, embed_model, k=k)
        all_hits.extend(hits)
        for h in hits:
            unique_snippets.add(h[:80])

    avg_docs   = len(all_hits) / len(eval_questions_gs)
    uniqueness = len(unique_snippets) / max(1, len(all_hits))
    topk_results.append({'top_k': k, 'avg_docs': avg_docs, 'uniqueness': uniqueness})
    print(f'  TOP_K={k:2d}  →  avg_docs={avg_docs:.1f}  uniqueness={uniqueness:.3f}')

topk_df = pd.DataFrame(topk_results)
topk_df['score'] = (topk_df['avg_docs']   / topk_df['avg_docs'].max() +
                    topk_df['uniqueness']  / topk_df['uniqueness'].max())
print('\nFull results (score = coverage + uniqueness, both normalised 0-1):')
display(topk_df.sort_values('score', ascending=False).reset_index(drop=True))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(topk_df['top_k'], topk_df['avg_docs'],   'o-', color='steelblue', lw=2)
axes[0].set_xlabel('TOP_K'); axes[0].set_ylabel('docs récupérés / requête')
axes[0].set_title('Couverture')

axes[1].plot(topk_df['top_k'], topk_df['uniqueness'], 'o-', color='darkorange', lw=2)
axes[1].set_xlabel('TOP_K'); axes[1].set_ylabel('unicité (0→1)')
axes[1].set_title('Diversité (unicité)')

axes[2].plot(topk_df['top_k'], topk_df['score'],      'o-', color='seagreen', lw=2)
axes[2].set_xlabel('TOP_K'); axes[2].set_ylabel('score combiné')
axes[2].set_title('Score (couverture + diversité)')
plt.tight_layout(); plt.show()

best_topk = int(topk_df.loc[topk_df['score'].idxmax(), 'top_k'])
print(f'\nBest TOP_K (coverage + diversity) : {best_topk}')
TOP_K = best_topk
print(f'TOP_K updated ✓ → {TOP_K}')

In [ ]:
# Évaluation : consommation de jetons par requête
rag_df = pd.DataFrame(rag_results)
print(f'Jetons totaux utilisés  : {rag_df["Tokens"].sum():,}')
print(f'Moyenne par requête     : {rag_df["Tokens"].mean():.0f}')
print(f'Min / Max               : {rag_df["Tokens"].min()} / {rag_df["Tokens"].max()}')

fig, ax = plt.subplots(figsize=(9, 3))
ax.barh([f'Q{i+1}' for i in range(len(rag_df))], rag_df['Tokens'], color='#6baed6')
ax.set_xlabel('jetons utilisés'); ax.set_title('RAG — consommation de jetons par requête')
plt.tight_layout(); plt.show()